# Autoencoder on plausibilitree-DL point clouds

Bare-bones PointNet-style **autoencoder** (PyTorch), unsupervised — it only
reads the feature files from `prepare.py` (labels are not used):

- `features/<plot_uid>.npz` — `points` (N, 3) float32 plus every extra point
  dimension (`semantic_code`, `tree_id`, `shrub_id`, `classification`,
  `branch_code`, `branch_id`, `height_above_ground`) and a `meta` JSON string

One plot = one sample. Pick the device (CPU / Apple GPU via MPS / CUDA), the
feature directory, the extra point channels to train on (`EXTRA_DIMS`), how
many plots to load (`N_SAMPLES`), the loss (`LOSS`), the model shape
(`N_POINTS`, `LATENT_DIM`), and the output range (`OUTPUT_RANGE`). Set
`N_SAMPLES = 1` to load a single sample and tweak the model until it
overfits it (then there is no validation split, so early stopping stays off).
Optional early stopping via `PATIENCE`/`MIN_DELTA` on the validation split.
The last cell plots original vs reconstructed points in the x-y, x-z, and
y-z projections.

Requires: `torch`, `numpy` (and `matplotlib` only for the optional plots).

In [ ]:
import glob
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

# ---- device: CPU / Apple integrated GPU (MPS) / CUDA cores ---------------
DEVICE_CHOICE = "auto"  # "auto" | "cpu" | "mps" | "cuda"


def pick_device(choice: str) -> torch.device:
    if choice in ("cuda", "auto") and torch.cuda.is_available():
        return torch.device("cuda")
    if choice in ("mps", "auto") and torch.backends.mps.is_available():
        return torch.device("mps")  # Apple integrated GPU
    if choice != "auto":
        print(f"warning: {choice!r} not available, falling back to CPU")
    return torch.device("cpu")


device = pick_device(DEVICE_CHOICE)
print("device:", device)

In [ ]:
# ---- user config -----------------------------------------------------------
FEATURES_DIR = "../dl_dataset/features"  # prepare.py -o <out>/features
EXTRA_DIMS = ["semantic_code"]           # extra per-point channels to train on,
                                         # stacked onto xyz (input = 3 + len(EXTRA_DIMS))
N_POINTS = 1024    # each cloud is padded/tiled or truncated to this many points
N_SAMPLES = 1      # how many plots to load: an int, or "all"; 1 = overfit one sample
LATENT_DIM = 128   # bottleneck size
LOSS = "mse"       # "mse" | "l1" | "huber" | "chamfer" (xyz Chamfer distance;
                   # note chamfer is O(N_POINTS^2) per sample)
OUTPUT_RANGE = "neg1_1"  # bound the decoder's xyz output to the normalized
                         # range: "neg1_1" -> tanh ([-1,1]), "zero_1" -> sigmoid
                         # ([0,1]), "none" -> linear. Match prepare.py's
                         # --normalize; extra dims are NOT bounded.
BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-3
SEED = 42
VAL_FRAC = 0.2     # validation fraction (only used when >1 sample is loaded)
PATIENCE = None    # early stopping on the val loss: stop after this many
                   # epochs without improvement; None = off
MIN_DELTA = 0.0    # minimum val-loss improvement that resets the patience counter

_first = sorted(glob.glob(os.path.join(FEATURES_DIR, "*.npz")))
if not _first:
    raise FileNotFoundError(f"no feature files in {FEATURES_DIR!r} - run prepare.py first")
with np.load(_first[0]) as z:
    print("available feature arrays:", z.files)

In [ ]:
class CloudDataset(Dataset):
    """One plot per sample: a (C, N_POINTS) feature matrix from the .npz
    (points + EXTRA_DIMS channels). Optionally limited to the first
    n_samples plots."""

    def __init__(self, features_dir, extra_dims, n_points, n_samples=None):
        self.n_points = n_points
        self.extra_dims = list(extra_dims)
        feats = sorted(glob.glob(os.path.join(features_dir, "*.npz")))
        if n_samples is not None:
            feats = feats[:n_samples]
        if not feats:
            raise FileNotFoundError(f"no feature files in {features_dir!r}")
        self.samples = feats
        self.names = [os.path.splitext(os.path.basename(f))[0] for f in feats]
        with np.load(feats[0]) as z:
            available = set(z.files)
        missing = [k for k in self.extra_dims if k not in available]
        if missing:
            raise KeyError(f"feature dims {missing} not in the .npz; "
                           f"available: {sorted(available)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        with np.load(self.samples[i]) as z:
            cols = [z["points"].astype(np.float32)]          # (N, 3)
            for k in self.extra_dims:
                cols.append(z[k].astype(np.float32)[:, None])  # (N, 1)
        mat = np.concatenate(cols, axis=1)                   # (N, C)
        # fix the point count: tile if short, truncate if long
        n = len(mat)
        if n >= self.n_points:
            mat = mat[: self.n_points]
        else:
            reps = int(np.ceil(self.n_points / n))
            mat = np.tile(mat, (reps, 1))[: self.n_points]
        return torch.from_numpy(mat.T.copy())                 # (C, N_POINTS)


dataset = CloudDataset(FEATURES_DIR, EXTRA_DIMS, N_POINTS,
                       n_samples=None if N_SAMPLES == "all" else N_SAMPLES)
print(f"loaded {len(dataset)} sample(s): {dataset.names}")
print(f"input channels = {3 + len(EXTRA_DIMS)} (xyz + {EXTRA_DIMS})")

In [ ]:
def chamfer_loss(a, b):
    """Symmetric Chamfer distance on the xyz channels. a, b: (B, C, N).
    O(N^2) memory per sample - keep N_POINTS modest."""
    pa = a[:, :3].transpose(1, 2)  # (B, N, 3)
    pb = b[:, :3].transpose(1, 2)
    d = torch.cdist(pa, pb)        # (B, N, N)
    return (d.min(dim=2).values.mean(dim=1) + d.min(dim=1).values.mean(dim=1)).mean()


class PointNetAE(nn.Module):
    """Bare-bones PointNet-style autoencoder: shared point MLP -> global max
    pool -> FC encoder to a latent vector -> FC decoder back to (C, N).
    The decoder's xyz channels (first 3) are bounded to `output_range`
    (tanh/sigmoid); extra dims pass through linearly."""

    def __init__(self, in_channels, n_points, latent_dim, output_range="none"):
        super().__init__()
        if output_range not in ("none", "neg1_1", "zero_1"):
            raise ValueError(f"output_range must be none|neg1_1|zero_1, got {output_range!r}")
        self.in_channels = in_channels
        self.n_points = n_points
        self.output_range = output_range
        self.enc = nn.Sequential(
            nn.Conv1d(in_channels, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(),
        )
        self.enc_fc = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, n_points * in_channels),
        )

    def forward(self, x):  # x: (B, C, N) -> recon (B, C, N), latent (B, L)
        h = torch.max(self.enc(x), dim=2).values  # (B, 1024)
        z = self.enc_fc(h)                        # (B, latent_dim)
        out = self.dec_fc(z).view(-1, self.in_channels, self.n_points)
        if self.output_range == "neg1_1":
            out = torch.cat([torch.tanh(out[:, :3]), out[:, 3:]], dim=1)
        elif self.output_range == "zero_1":
            out = torch.cat([torch.sigmoid(out[:, :3]), out[:, 3:]], dim=1)
        return out, z


def make_criterion(name):
    if name == "mse":
        return nn.MSELoss()
    if name == "l1":
        return nn.L1Loss()
    if name == "huber":
        return nn.HuberLoss()
    if name == "chamfer":
        return chamfer_loss
    raise ValueError(f"unknown LOSS {name!r} (mse | l1 | huber | chamfer)")


model = PointNetAE(in_channels=3 + len(EXTRA_DIMS), n_points=N_POINTS,
                   latent_dim=LATENT_DIM, output_range=OUTPUT_RANGE).to(device)
print(model)

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

n_val = max(1, int(round(len(dataset) * VAL_FRAC))) if len(dataset) > 1 else 0
n_train = len(dataset) - n_val
train_set, val_set = random_split(dataset, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(SEED))
# drop_last avoids a final 1-sample batch, which BatchNorm1d rejects in train mode
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=len(train_set) > BATCH_SIZE)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE) if n_val else None

criterion = make_criterion(LOSS)
opt = torch.optim.Adam(model.parameters(), lr=LR)

best_val = float("inf")
bad = 0
best_state = None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    mse_total = 0.0
    n = 0
    for x in train_loader:
        x = x.to(device)
        if x.shape[0] == 1:
            # single-sample overfit mode: BatchNorm needs >1 row per batch
            x = x.repeat(BATCH_SIZE, 1, 1)
        opt.zero_grad()
        recon, _ = model(x)
        loss = criterion(recon, x)
        loss.backward()
        opt.step()
        total += loss.item() * x.shape[0]
        mse_total += nn.functional.mse_loss(recon, x).item() * x.shape[0]
        n += x.shape[0]
    msg = f"epoch {epoch:3d}/{EPOCHS}  {LOSS} {total / max(n, 1):.6f}  MSE {mse_total / max(n, 1):.6f}"
    if val_loader is not None:
        model.eval()
        val_total = 0.0
        vn = 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                recon, _ = model(x)
                val_total += criterion(recon, x).item() * x.shape[0]
                vn += x.shape[0]
        val_loss = val_total / max(vn, 1)
        msg += f"  val {LOSS} {val_loss:.6f}"
        if PATIENCE is not None:
            if val_loss < best_val - MIN_DELTA:
                best_val = val_loss
                bad = 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad > PATIENCE:
                    print(f"early stopping at epoch {epoch} "
                          f"(val {LOSS} no improvement for {PATIENCE} epochs)")
                    break
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS or "early stopping" in msg:
        print(msg)

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"restored best model (val {LOSS} {best_val:.6f})")
elif PATIENCE is not None:
    print("PATIENCE set but no validation split (single sample) - early stopping inactive")

In [ ]:
# ---- reconstruct the first sample and report the error --------------------
model.eval()
with torch.no_grad():
    x = dataset[0][None].to(device)  # (1, C, N); eval mode so BatchNorm is fine at B=1
    recon, z = model(x)
    obj = criterion(recon, x).item()
    mse = nn.functional.mse_loss(recon, x).item()
print(f"sample {dataset.names[0]}: {LOSS} {obj:.6f}, MSE {mse:.6f}, latent {tuple(z.shape)}")
# both scatter series below are in this same model space - if the plot axes
# fall far outside this range, something is wrong; otherwise it's model fit
print(f"input value range: [{float(x.min()):.4g}, {float(x.max()):.4g}] (model space)")
print(f"recon xyz range: [{float(recon[0, :3].min()):.4g}, {float(recon[0, :3].max()):.4g}] "
      f"(bounded by OUTPUT_RANGE={OUTPUT_RANGE!r})")

try:
    import matplotlib.pyplot as plt
    o = x[0, :3].cpu().numpy()        # original xyz points
    r = recon[0, :3].cpu().numpy()    # reconstructed xyz points
    pairs = [(0, 1, "x", "y"), (0, 2, "x", "z"), (1, 2, "y", "z")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (i, j, li, lj) in zip(axes, pairs):
        ax.scatter(o[i], o[j], s=4, label="original")
        ax.scatter(r[i], r[j], s=4, label="reconstructed")
        ax.set_xlabel(li)
        ax.set_ylabel(lj)
        ax.legend()
    fig.suptitle(f"reconstruction of {dataset.names[0]}")
    plt.show()
except ImportError:
    print("matplotlib not installed - skipping plots")